### Sampling input spaces
Import libraries

In [ ]:
import sys

libs = {
  "./../modules",
  "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
import reading as read_mod
import sampling as smp_mod
import plotting as plt_mod
import clustering as clst_mod

Define inputs
> Data

In [ ]:
# Path to data
path_to_dset = "./../output/lowrank_state/dset.p"
# Reduced dimensions (mass fractions)
rdim = 5
# Columns
cols = ["rho"] + [f"z{i+1}" for i in range(rdim)]
cols_clst = cols + ["Th", "dT"]
cols_smpl = cols + ["Th", "Te", "dT"]
# Variables
varnames = {
  "rho": r"$\rho$",
  "Th": r"$T_h$",
  "Te": r"$T_e$",
  "dT": r"$dT$"
}
varnames.update({f"z{i+1}": f"$z_{i+1}$" for i in range(rdim)})

> Sampling

In [ ]:
# Minimum value
eps = 1e-7
# Bounds
path_to_bounds = "./../output/lowrank_state/bounds.json"
bounds_only_phys = False
bounds_extra_perc = 0.1
scaling = "std"
# KDE
kde_samples_perc = 0.33
kde_bandwidth = 0.4
# Nb. data points: in-domain conditions
nb_train_cases = 3000
nb_test_cases = 100
# Nb. data points
nb_samples = {
  "train": nb_train_cases,
  "valid": nb_train_cases,
  "test": nb_test_cases
}
# Noise
noise_std = 0.02

> Sampling: transformation

In [ ]:
def xform_dir(x):
  x[:,0] = np.log(x[:,0])
  return x

def xform_inv(x):
  x[:,0] = np.exp(x[:,0])
  return x

xform = {"dir": xform_dir, "inv": xform_inv}

> Clustering

In [ ]:
clustering = "gm"
n_clusters = 1
kde_samples_perc = [kde_samples_perc] * n_clusters

> Saving

In [ ]:
# Saving path
path_to_save = "./../output/sampling/"
path_to_save = os.path.abspath(path_to_save)
os.makedirs(path_to_save, exist_ok=True)

Load data

In [ ]:
# Dataset
dset = read_mod.load_dataset(path_to_dset)
# Bounds
with open(path_to_bounds) as file:
  bounds_phys = json.load(file)
bounds_phys = {k: bounds_phys[k] for k in cols_smpl}

Conditional sampling

In [ ]:
# Set physical conditions
conds_phys = smp_mod.get_conditions(
  bounded_qoi=list(bounds_phys.keys()),
  sampled_qoi=cols_smpl
)

**Clustering \& Sampling**

Initialize data matrices \& remove closest points

In [ ]:
X = np.vstack([dset[k] for k in cols_clst]).T
X = pd.DataFrame(X, columns=cols_clst)

# Scale dataset
if (scaling == "std"):
  Xs = (X - X.mean()) / X.std()
elif (scaling == "minmax"):
  Xs = (X - X.min()) / (X.max() - X.min())
else:
  Xs = None

# Remove closest points
if (Xs is not None):
  indices = np.unique(Xs.round(decimals=4), return_index=True, axis=0)[1]
  X = X.to_numpy()[indices]

Cluster manifold

In [ ]:
clust = clst_mod.Clusterer(
  X,
  scaling="std",
  clustering=clustering,
  n_clusters=n_clusters
)
X0 = pd.DataFrame(clust.centers, columns=cols_clst)

In [ ]:
lbl = clust.cluster(X)
Xc = [X[lbl == i] for i in range(n_clusters)]

In [ ]:
smp = []
for (i, x) in enumerate(Xc):
  x = pd.DataFrame(x, columns=cols_clst)
  x["Te"] = x["Th"] - x["dT"]
  x = x[cols_smpl].to_numpy()
  smp.append(smp_mod.Sampler(
    X=x,
    cols=cols_smpl,
    xform=xform,
    conds_phys=conds_phys,
    bounds_phys=bounds_phys,
    bounds_only_phys=bounds_only_phys,
    bounds_extra_perc=bounds_extra_perc,
    kde_samples_perc=kde_samples_perc[i],
    kde_bandwidth=kde_bandwidth,
    noise_std=noise_std
  ))

In [ ]:
# Compute overall bounds
bounds = np.stack([s.bounds for s in smp], axis=0)
bounds = np.stack([
  np.amin(bounds[:,0,:], axis=0),
  np.amax(bounds[:,1,:], axis=0)
], axis=0)
bounds = pd.DataFrame(bounds, columns=cols_smpl)
bounds = bounds[cols_clst].to_numpy()
# Save overall bounds
np.savetxt(path_to_save + "/bounds.txt", bounds.T)

Uniform and kde-based sampling

In [ ]:
index = X0.index.tolist()
pts_perc = [1/n_clusters] * n_clusters

In [ ]:
idx_perc = {i: p for (i, p) in list(zip(index, pts_perc))}

In [ ]:
sampled_data = {}
for k, nb in nb_samples.items():
  samples_k = []
  for i in range(n_clusters):
    samples_ki = smp[i].sample_lhs_kde(int(np.ceil(nb*idx_perc[i])))
    samples_ki = pd.DataFrame(samples_ki, columns=cols_smpl)
    samples_ki = samples_ki[cols_clst].to_numpy()
    samples_k.append(samples_ki)
  sampled_data[k] = samples_k
data = {k: np.vstack(sampled_data[k]) for k in nb_samples.keys()}

In [ ]:
print("Data shapes:")
tf.nest.map_structure(lambda x: x.shape, data)

Save samples in files

In [ ]:
# Saving
for (k, samples) in data.items():
  df_smp = pd.DataFrame(samples, columns=cols_clst)
  df_smp.to_csv(path_to_save + f"/{k}.csv", float_format="%.8e", index=False)

#### Postprocessing
Normilized between 0 and 1

In [ ]:
Xs = np.vstack(sampled_data["train"])
xlim = Xs.min(axis=0), Xs.max(axis=0)
xlim = np.vstack(xlim)

Piecewise plot

In [ ]:
fn = lambda x: x.T
# Names
names = [varnames[k] for k in cols_clst]
# Data
plt_mod.plot_pairwise(
  X=tf.nest.map_structure(fn, Xc),
  names=names,
  xlim=xlim,
  figsize=(10.0, 10.0),
  labelpad=[20,10],
  figname=path_to_save + "/piecewise_data.png",
  show=False
)
# Samples
plt_mod.plot_pairwise(
  X=tf.nest.map_structure(fn, sampled_data["train"]),
  names=names,
  xlim=xlim,
  figsize=(10.0, 10.0),
  labelpad=[20,10],
  figname=path_to_save + "/piecewise_samples.png",
  show=False
)